# begin from this notebook 

In [22]:
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from scipy.optimize import minimize
import numpy as np

In [2]:
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

passes = [
    {"name": "pass_0", "sat": "SAT_A", "priority": 5, "energy": 3},
    {"name": "pass_1", "sat": "SAT_A", "priority": 4, "energy": 2},
    {"name": "pass_2", "sat": "SAT_B", "priority": 3, "energy": 4},
    {"name": "pass_3", "sat": "SAT_C", "priority": 6, "energy": 1},
]

qp = QuadraticProgram(name="Satellite_Tasking")
for p in passes:
    qp.binary_var(name=p["name"])

qp.maximize(linear={p["name"]: p["priority"] for p in passes})

sat_capacity = {"SAT_A": 3, "SAT_B": 4, "SAT_C": 1}
for sat, cap in sat_capacity.items():
    terms = {p["name"]: p["energy"] for p in passes if p["sat"] == sat}
    if terms:
        qp.linear_constraint(linear=terms, sense="<=", rhs=cap, name=f"battery_{sat}")

qp.linear_constraint(linear={"pass_0": 1, "pass_1": 1},
                     sense="<=", rhs=1, name="window_overlap_A")

qp.linear_constraint(linear={"pass_2": 1, "pass_3": 1},
                     sense="<=", rhs=1, name="antenna_conflict")

to_qubo = QuadraticProgramToQubo()
qubo = to_qubo.convert(qp)
print("Penalty:", to_qubo.penalty)
print(qubo.prettyprint())

Penalty: 57.0
Problem name: Satellite_Tasking

Minimize
  57*battery_SAT_A@int_slack@0^2
  + 228*battery_SAT_A@int_slack@0*battery_SAT_A@int_slack@1
  + 228*battery_SAT_A@int_slack@1^2 + 57*battery_SAT_B@int_slack@0^2
  + 228*battery_SAT_B@int_slack@0*battery_SAT_B@int_slack@1
  + 114*battery_SAT_B@int_slack@0*battery_SAT_B@int_slack@2
  + 228*battery_SAT_B@int_slack@1^2
  + 228*battery_SAT_B@int_slack@1*battery_SAT_B@int_slack@2
  + 57*battery_SAT_B@int_slack@2^2 + 57*battery_SAT_C@int_slack@0^2
  + 342*pass_0*battery_SAT_A@int_slack@0 + 684*pass_0*battery_SAT_A@int_slack@1
  + 513*pass_0^2 + 703*pass_0*pass_1 + 228*pass_1*battery_SAT_A@int_slack@0
  + 456*pass_1*battery_SAT_A@int_slack@1 + 228*pass_1^2
  + 456*pass_2*battery_SAT_B@int_slack@0 + 912*pass_2*battery_SAT_B@int_slack@1
  + 456*pass_2*battery_SAT_B@int_slack@2 + 912*pass_2^2 + 19*pass_2*pass_3
  + 114*pass_3*battery_SAT_C@int_slack@0 + 57*pass_3^2
  - 342*battery_SAT_A@int_slack@0 - 684*battery_SAT_A@int_slack@1
  - 456*ba

In [3]:
from itertools import product

# get QUBO as an Ising-like sparse list
qubo_dict = qubo.objective.linear.to_dict()
quad = qubo.objective.quadratic.to_dict()
const = qubo.objective.constant

def energy(x):
    e = const
    for i, c in qubo_dict.items():
        e += c * x[i]
    for (i, j), c in quad.items():
        e += c * x[i] * x[j]
    return e

best = None
for x in product([0, 1], repeat=qubo.get_num_vars()):
    e = energy(x)
    if best is None or e < best[0]:
        best = (e, x)

print("Best energy:", best[0])
print("Best bitstring:", best[1])
print("Variable names:", qubo.variables_index)

Best energy: -11.0
Best bitstring: (1, 0, 0, 1, 0, 0, 1, 1, 1, 0)
Variable names: {'pass_0': 0, 'pass_1': 1, 'pass_2': 2, 'pass_3': 3, 'battery_SAT_A@int_slack@0': 4, 'battery_SAT_A@int_slack@1': 5, 'battery_SAT_B@int_slack@0': 6, 'battery_SAT_B@int_slack@1': 7, 'battery_SAT_B@int_slack@2': 8, 'battery_SAT_C@int_slack@0': 9}


In [4]:
from qiskit.quantum_info import SparsePauliOp

# Convert QUBO to Ising Hamiltonian
ising, offset = qubo.to_ising()
print("Number of qubits:", ising.num_qubits)
print("Offset:", offset)
print("Hamiltonian terms:", len(ising))

Number of qubits: 10
Offset: 656.0
Hamiltonian terms: 20


In [11]:
def build_satellite_qubo(n_passes=8, n_sats=3, seed=42):
    rng = np.random.default_rng(seed)
    passes = []
    for i in range(n_passes):
        passes.append({
            "name": f"pass_{i}",
            "sat": f"SAT_{chr(65 + i % n_sats)}",
            "priority": int(rng.integers(3, 10)),
            "energy": int(rng.integers(1, 4)),
        })

    qp = QuadraticProgram(name=f"Satellite_N{n_passes}")
    for p in passes:
        qp.binary_var(name=p["name"])
    qp.maximize(linear={p["name"]: p["priority"] for p in passes})

    for s in sorted({p["sat"] for p in passes}):
        terms = {p["name"]: p["energy"] for p in passes if p["sat"] == s}
        cap = sum(terms.values()) // 2
        qp.linear_constraint(linear=terms, sense="<=", rhs=cap, name=f"battery_{s}")

    pairs = set()
    while len(pairs) < n_passes // 2:
        i, j = sorted(rng.choice(n_passes, 2, replace=False))
        pairs.add((i, j))
    for i, j in pairs:
        qp.linear_constraint(
            linear={passes[i]["name"]: 1, passes[j]["name"]: 1},
            sense="<=", rhs=1, name=f"conflict_{i}_{j}",
        )

    to_qubo = QuadraticProgramToQubo()
    return qp, to_qubo.convert(qp)

In [12]:
for n in [4, 6, 8]:
    qp, qubo = build_satellite_qubo(n_passes=n)
    ising, offset = qubo.to_ising()
    print(f"n_passes={n:2d}  qubits={ising.num_qubits:3d}  terms={len(ising):4d}")

n_passes= 4  qubits=  8  terms=  17
n_passes= 6  qubits= 11  terms=  29
n_passes= 8  qubits= 15  terms=  49


In [19]:
#x
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import VarQITE
from qiskit_algorithms.time_evolvers.variational import ImaginaryMcLachlanPrinciple
from qiskit_algorithms.time_evolvers import TimeEvolutionProblem
from qiskit.primitives import StatevectorEstimator

# Build the 8-qubit instance
qp, qubo = build_satellite_qubo(n_passes=4)
ising, offset = qubo.to_ising()
num_qubits = ising.num_qubits
print(f"Solving {num_qubits}-qubit MIS problem")

# Variational ansatz
ansatz = efficient_su2(num_qubits, reps=2)
init_params = np.random.default_rng(0).uniform(-0.1, 0.1, ansatz.num_parameters)

# VarQITE setup
var_principle = ImaginaryMcLachlanPrinciple()
evolution_problem = TimeEvolutionProblem(ising, time=5.0)

var_qite = VarQITE(ansatz, init_params, var_principle, StatevectorEstimator())
result = var_qite.evolve(evolution_problem)
print("Evolution done.")
print("Final parameters (first 3):", result.parameter_values[:3])

Solving 8-qubit MIS problem
Evolution done.
Final parameters (first 3): [[ 2.73923375e-02 -4.60426572e-02 -9.18052952e-02 -9.66944729e-02
   6.26540478e-02  8.25511155e-02  2.13271552e-02  4.58993122e-02
   8.72499829e-03  8.70144848e-02  6.31707108e-02 -9.94523000e-02
   7.14808553e-02 -9.32828849e-02  4.59310893e-02 -6.48688759e-02
   7.26357845e-02  8.29224405e-03 -4.00576219e-02 -1.54625558e-02
  -9.43360658e-02 -7.51433447e-02  3.41248829e-02  2.94379023e-02
   2.30770223e-02 -2.32644891e-02  9.94419872e-02  9.61670678e-02
   3.71083969e-02  3.00918553e-02  3.76893461e-02 -2.22157152e-02
  -7.29806990e-02  4.42976680e-02  5.07086450e-03 -3.79516249e-02
  -2.83292823e-03  7.78975669e-02  8.68087032e-02 -2.84409607e-02
   1.43059661e-02 -3.56261218e-02  1.88600060e-02 -3.24177549e-02
  -2.16761999e-02  7.80548704e-02 -5.45684813e-02  2.46374289e-02]
 [ 1.24350083e-01 -2.94623248e-01 -2.03137410e-01 -3.73687264e-01
   2.58721986e-01  5.21449823e-01  2.29234203e-02  7.52139601e-02
   

In [20]:
#this approach didn't work well with VarQITE
#switched to another approach QAOA, in next cell
from qiskit.quantum_info import Statevector

# Final parameters from VarQITE result
final_params = result.parameter_values[-1]

# Build the statevector of the evolved ansatz
evolved_circuit = ansatz.assign_parameters(final_params)
sv = Statevector(evolved_circuit)

# Sample probabilities
probs = sv.probabilities_dict()
top = sorted(probs.items(), key=lambda kv: -kv[1])[:10]

print("Top-10 measured bitstrings (bitstring, probability):")
for bits, p in top:
    print(bits, f"{p:.4f}")

Top-10 measured bitstrings (bitstring, probability):
00110100 0.0604
00111011 0.0475
00011011 0.0425
00011100 0.0311
01011011 0.0296
00111100 0.0276
00001111 0.0267
00010111 0.0249
01011100 0.0224
00011000 0.0219


In [26]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization import QuadraticProgram

# ---- Rebuilding benchmark ----
qp_bench, qubo_bench = build_satellite_qubo(n_passes=4)
qubo_qp = qubo_bench

# ---- Step 1: greedy seed + continuous relaxation ----
def qubo_obj(x):
    val = qubo_qp.objective.constant
    for i, c in qubo_qp.objective.linear.to_dict().items():
        val += c * x[i]
    for (i,j), c in qubo_qp.objective.quadratic.to_dict().items():
        val += c * x[i] * x[j]
    return val

n_vars = qubo_qp.get_num_vars()

# Greedy approach: sort pass variables by priority, only taking if feasible
pass_priorities = [(0, 5), (1, 4), (2, 3), (3, 6)]
greedy = np.zeros(n_vars)
taken = []
for idx, pri in sorted(pass_priorities, key=lambda t: -t[1]):
    if idx not in taken:
        greedy[idx] = 1.0
        taken.append(idx)
        # apply conflict rule
        if idx == 0: taken.append(1)
        if idx == 2: taken.append(3)
print("Greedy seed:", greedy[:4])

res = minimize(qubo_obj, x0=greedy, method='L-BFGS-B',
               bounds=[(0,1)]*n_vars)
c_star = res.x
print("Relaxed solution:", np.round(c_star, 3))


# ---- Step 2: regularizing and building warm-start state ----
epsilon = 0.25
c_clipped = np.clip(c_star, epsilon, 1 - epsilon)
thetas = 2 * np.arcsin(np.sqrt(c_clipped))

init_qc = QuantumCircuit(n_vars)
for i, theta in enumerate(thetas):
    init_qc.ry(theta, i)

# ---- Step 3: customizing mixer RY(theta) RZ(-2β) RY(-theta) ----
beta = Parameter("β")
ws_mixer = QuantumCircuit(n_vars)
for i in range(n_vars):
    ws_mixer.ry(thetas[i], i)
    ws_mixer.rz(-2*beta, i)
    ws_mixer.ry(-thetas[i], i)

# ---- Step 4: running warm-start QAOA ----
sampler = StatevectorSampler(seed=42)
qaoa_ws = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=200),
               reps=2, initial_state=init_qc, mixer=ws_mixer,
               initial_point=[0.5, 0.5, 0.5, 0.5])
solver = MinimumEigenOptimizer(qaoa_ws)
result_ws = solver.solve(qp_bench)

print("\n=== Warm-Start QAOA Result ===")
print("Optimal value:", result_ws.fval)
print("Optimal bitstring:", result_ws.x)

Greedy seed: [1. 0. 1. 1.]
Relaxed solution: [1.    0.515 0.339 0.    0.    0.    0.    0.   ]

=== Warm-Start QAOA Result ===
Optimal value: 3.0
Optimal bitstring: [1. 0. 0. 0.]


In [28]:
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer

qp_bench, _ = build_satellite_qubo(n_passes=4)

for penalty in [8.0, 15.0, 25.0]:
    to_qubo = QuadraticProgramToQubo(penalty=penalty)
    qubo_small = to_qubo.convert(qp_bench)
    ising_small, _ = qubo_small.to_ising()
    sampler = StatevectorSampler(seed=42)
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=4)
    solver = MinimumEigenOptimizer(qaoa)
    r = solver.solve(qp_bench)  # solve original problem for correct decoding
    print(f"penalty={penalty}  fval={r.fval}  x={r.x}")

/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


penalty=8.0  fval=3.0  x=[1. 0. 0. 0.]


/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


penalty=15.0  fval=3.0  x=[1. 0. 0. 0.]


/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


penalty=25.0  fval=3.0  x=[1. 0. 0. 0.]


In [29]:
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# Auditing the exact object pass to solvers
qp_bench, _ = build_satellite_qubo(n_passes=4)
print("Variables:", [v.name for v in qp_bench.variables])
print("Objective at [1,0,0,0]:", qp_bench.objective.evaluate([1, 0, 0, 0]))

# Exacting solver through the same pipeline
exact_optimizer = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact_optimizer.solve(qp_bench)
print("Exact x:", exact_result.x)
print("Exact fval:", exact_result.fval)

Variables: ['pass_0', 'pass_1', 'pass_2', 'pass_3']
Objective at [1,0,0,0]: 3.0
Exact x: [1. 0. 0. 0.]
Exact fval: 3.0


In [31]:
def build_satellite_qubo(n_passes=4, n_sats=3, seed=42):
    # FIXED priorities for reproducible benchmark
    fixed_priorities = [5, 4, 3, 6, 7, 5, 8, 6]
    fixed_energies   = [3, 2, 4, 1, 3, 2, 2, 3]

    passes = []
    for i in range(n_passes):
        passes.append({
            "name": f"pass_{i}",
            "sat": f"SAT_{chr(65 + i % n_sats)}",
            "priority": fixed_priorities[i % len(fixed_priorities)],
            "energy":   fixed_energies[i % len(fixed_energies)],
        })
    
    qp = QuadraticProgram(name="Satellite_4Q")
    priorities = {"pass_0": 5, "pass_1": 4, "pass_2": 3, "pass_3": 6}
    energies   = {"pass_0": 3, "pass_1": 2, "pass_2": 4, "pass_3": 1}
    sat_map    = {"pass_0": "SAT_A", "pass_1": "SAT_A", "pass_2": "SAT_B", "pass_3": "SAT_C"}
    caps       = {"SAT_A": 3, "SAT_B": 4, "SAT_C": 1}

    for name in priorities:
        qp.binary_var(name=name)
    qp.maximize(linear=priorities)

    # Conflict penalties (no slack needed)
    conflicts = [("pass_0", "pass_1"), ("pass_2", "pass_3")]
    for a, b in conflicts:
        qp.linear_constraint(linear={a: 1, b: 1}, sense="<=", rhs=1, name=f"c_{a}_{b}")

    to_qubo = QuadraticProgramToQubo()
    return qp, to_qubo.convert(qp)

In [34]:
qp_bench, _ = build_satellite_qubo(n_passes=4)
print("Priorities:", qp_bench.objective.linear.to_dict(use_name=True))
print("Objective at [1,0,0,0]:", qp_bench.objective.evaluate([1, 0, 0, 0]))

from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp_bench)
print("Exact x:", exact.x)
print("Exact fval:", exact.fval)

Priorities: {'pass_0': np.float64(5.0), 'pass_1': np.float64(4.0), 'pass_2': np.float64(3.0), 'pass_3': np.float64(6.0)}
Objective at [1,0,0,0]: 5.0
Exact x: [1. 0. 0. 1.]
Exact fval: 11.0


In [35]:
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer
import numpy as np

qp_bench, _ = build_satellite_qubo(n_passes=4)

best = None
for seed in range(5):
    sampler = StatevectorSampler(seed=seed)
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=3,
                initial_point=np.random.RandomState(seed).uniform(0, np.pi, 6))
    r = MinimumEigenOptimizer(qaoa).solve(qp_bench)
    print(f"seed={seed} fval={r.fval} x={r.x}")
    if best is None or r.fval > best.fval:
        best = r

print("\nBEST:", best.fval, best.x)

seed=0 fval=11.0 x=[1. 0. 0. 1.]
seed=1 fval=11.0 x=[1. 0. 0. 1.]
seed=2 fval=11.0 x=[1. 0. 0. 1.]
seed=3 fval=11.0 x=[1. 0. 0. 1.]
seed=4 fval=11.0 x=[1. 0. 0. 1.]

BEST: 11.0 [1. 0. 0. 1.]


In [38]:
def get_benchmark():
    qp, qubo = build_satellite_qubo(n_passes=4)
    return qp, qubo

In [37]:
def decode_and_validate(result, qp):
    decisions = dict(zip([v.name for v in qp.variables], result.x))
    priority = sum(qp.objective.linear.to_dict(use_name=True).get(k, 0) * v
                   for k, v in decisions.items())
    feasible = all(
        sum(c.linear.to_dict(use_name=True).get(k, 0) * decisions[k]
            for k in c.linear.to_dict(use_name=True)) <= c.rhs
        for c in qp.linear_constraints
    )
    return decisions, priority, feasible

In [39]:
def decode_and_validate(result, qp):
    decisions = {v.name: int(round(result.x[i]))
                 for i, v in enumerate(qp.variables)}
    priorities = qp.objective.linear.to_dict(use_name=True)
    score = sum(priorities.get(k, 0) * v for k, v in decisions.items())
    feasible = True
    for c in qp.linear_constraints:
        lhs = sum(c.linear.to_dict(use_name=True).get(k, 0) * decisions[k]
                  for k in c.linear.to_dict(use_name=True))
        if lhs > c.rhs:
            feasible = False
    return decisions, score, feasible

In [40]:
qp_bench, _ = get_benchmark()
decisions, score, ok = decode_and_validate(best, qp_bench)
print("Schedule:", decisions)
print("Score:", score, "| Feasible:", ok)

Schedule: {'pass_0': 1, 'pass_1': 0, 'pass_2': 0, 'pass_3': 1}
Score: 11.0 | Feasible: True


 # This notebook documents two findings that shaped the final approach:

1. **Benchmark drift bug.** An earlier version of `build_satellite_qubo`
   regenerated random priorities on every call, so each solver ran on a
   different problem. This was diagnosed by comparing the exact solver
   against the QAOA result.

2. **VarQITE failure.** Simulated imaginary-time evolution produced a
   near-uniform distribution (top bitstring probability ~0.06) and did
   not recover the optimum. This motivated the switch to QAOA.